# Notebook 01 — Acquisition des Donnees

**Projet :** Prediction du risque d'abandon scolaire  
**Equipe :** Hugo RAGUIN · Amine TALEB · Elliot FIORESE  

---

## Contexte metier

Le domaine d'etude est l'**analytics educatif**. Un etablissement d'enseignement dispose souvent de donnees fragmentees : notes continues, absences, retards dans les remises, connexions a une plateforme pedagogique (LMS), statut boursier, parcours anterieur et indicateurs socio-economiques. Isolement, ces variables sont peu actionnables ; combinees, elles permettent de detecter des **signaux faibles de decrochage**.

**Problematique :** Identifier le plus tot possible les etudiants a risque d'abandon ou d'echec academique afin de declencher des actions ciblees : tutorat, suivi pedagogique, accompagnement social ou adaptation du rythme d'apprentissage.

**Formulation :** Classification binaire supervisee — variable cible `dropout_risk` (True / False).

## 1. Configuration de l'environnement

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath('..')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from student_risk_dataset import generate_student_dataset, ensure_student_dataset
from data_clean import load_raw_data
from utils_viz import set_custom_style

set_custom_style(theme='light')
%matplotlib inline
print('Environnement pret. Repertoire de travail :', os.getcwd())

## 2. Justification du jeu de donnees

### Source choisie : Dataset etudiant synthetique reproductible

Nous avons genere un dataset synthetique realiste plutot que d'utiliser un dataset Kaggle generique pour les raisons suivantes :

| Critere | Dataset synthetique (choix retenu) | Kaggle generique |
|---|---|---|
| **Alignement metier** | Variables calquees sur un SI etudiant reel | Variables souvent non pertinentes pour l'analytics educatif |
| **Reproductibilite** | Graine aleatoire fixe (`seed=42`) | Dependant des mise a jour Kaggle |
| **Confidentialite** | Aucune donnee personnelle reelle | Risque de donnees identifiantes |
| **Coherence** | Relations inter-variables calibrees (assiduite -> notes) | Correlations parfois artificielles |
| **Extensibilite** | Remplacable par un jeu reel anonymise | Difficile a remplacer sans recoder le pipeline |

Le dataset simule **1 600 etudiants** avec **16 variables** couvrant quatre dimensions :
- **Profil socio-demograhique** : age, programme, semestre, statut boursier, education parentale
- **Comportement academique** : assiduite, retards, sessions LMS, heures de travail
- **Performance academique** : moyenne anterieure, evaluation continue
- **Contexte de vie** : trajet domicile-campus, acces internet, stress

## 3. Chargement et premier regard

In [ ]:
# Generation et chargement du dataset brut (reproductible, seed=42)
raw_path = ensure_student_dataset(force=True)
df = load_raw_data(str(raw_path))

print(f'Dimensions : {df.shape[0]} etudiants x {df.shape[1]} variables')
df.head()

## 4. Description du dictionnaire de variables

| Variable | Type | Description | Plage |
|---|---|---|---|
| `student_id` | Categoriel | Identifiant unique etudiant | STD-00001 … STD-01600 |
| `program` | Categoriel | Filiere d'etudes | 4 programmes |
| `semester` | Entier | Semestre en cours | 1 – 6 |
| `age` | Numerique | Age de l'etudiant | 17 – 34 ans |
| `scholarship` | Booleen | Statut boursier | True / False |
| `parental_education` | Categoriel | Niveau d'education des parents | secondary / undergraduate / graduate |
| `internet_access` | Booleen | Acces internet a domicile | True / False |
| `commute_minutes` | Numerique | Duree trajet domicile-campus | 5 – 95 min |
| `study_hours_per_week` | Numerique | Heures de travail personnel / semaine | 2 – 28 h |
| `lms_sessions_week` | Numerique | Sessions LMS par semaine | 1 – 24 |
| `attendance_rate` | Numerique | Taux d'assiduite | 35 – 100 % |
| `assignment_delay_days` | Numerique | Retard moyen de remise | 0 – 12 jours |
| `prior_average` | Numerique | Moyenne des semestres anterieurs | 2 – 19.5 / 20 |
| `continuous_assessment` | Numerique | Note d'evaluation continue | 0 – 20 |
| `stress_index` | Numerique | Indice de stress auto-declare | 10 – 95 |
| `dropout_risk` | Booleen | **Cible** : risque d'abandon detecte | True / False |

In [ ]:
# Types et completude par variable
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'n_missing': df.isnull().sum(),
    'pct_missing': (df.isnull().mean() * 100).round(2),
    'n_unique': df.nunique()
})
print(info_df.to_string())

## 5. Analyse de la variable cible

La variable cible `dropout_risk` est **desequilibree** (~8-9 % de positifs). Ce desequilibre est realiste (les abandons sont minoritaires dans un etablissement) mais implique des precautions methodologiques :
- Split **stratifie** pour preserver la proportion dans chaque partition
- **`class_weight='balanced'`** dans les modeles
- Priorite au **rappel** plutot qu'a l'accuracy

In [ ]:
# Distribution de la cible
risk_counts = df['dropout_risk'].value_counts()
risk_pct = df['dropout_risk'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Camembert
labels = ['Non a risque', 'A risque']
sizes = [risk_counts.get(False, 0), risk_counts.get(True, 0)]
colors_pie = ['#188038', '#D93025']
axes[0].pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 10})
axes[0].set_title('Repartition de la variable cible')

# Barplot par programme
prog_risk = df.groupby('program')['dropout_risk'].mean() * 100
colors_bar = ['#D93025' if v > 10 else '#F29900' if v > 7 else '#188038'
              for v in prog_risk.values]
axes[1].bar(prog_risk.index, prog_risk.values, color=colors_bar, edgecolor='none')
axes[1].set_ylabel('Taux de risque (%)')
axes[1].set_title('Taux de risque par programme')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(prog_risk.values):
    axes[1].text(i, v + 0.2, f'{v:.1f}%', ha='center', fontsize=9)

fig.suptitle('Distribution de la variable cible dropout_risk', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

print(f'Etudiants a risque : {sizes[1]} / {sum(sizes)} ({risk_pct.get(True, 0):.1f} %)')
print(f'Ratio classe majoritaire / minoritaire : {sizes[0]/sizes[1]:.1f}:1')

In [ ]:
# Apercu des statistiques de base
numeric_cols = [
    'age', 'study_hours_per_week', 'lms_sessions_week',
    'attendance_rate', 'prior_average', 'continuous_assessment', 'stress_index'
]
print('=== Statistiques descriptives du dataset brut ===')
df[numeric_cols].describe().round(2)

In [ ]:
# Repartition par programme et semestre
print('=== Repartition par programme ===')
prog_counts = df['program'].value_counts()
for prog, cnt in prog_counts.items():
    print(f'  {prog:22s} : {cnt:4d} etudiants ({cnt/len(df)*100:.1f} %)')

print()
print('=== Repartition par semestre ===')
sem_counts = df['semester'].value_counts().sort_index()
for sem, cnt in sem_counts.items():
    print(f'  Semestre {sem} : {cnt:4d} etudiants ({cnt/len(df)*100:.1f} %)')

## 6. Bilan de l'acquisition

- **1 600 observations** et **16 variables** chargees avec succes
- Valeurs manquantes presentes sur 10 variables (taux entre 2 % et 4 %) — traitement en Notebook 02
- Variable cible `dropout_risk` : **8-9 % de positifs** — dataset desequilibre, pris en compte dans la modelisation
- Aucun doublon detecte (identifiants uniques par construction)
- Le pipeline est reproductible : meme `seed=42` => meme dataset => memes resultats